# 🧠 Entropy-Driven CfC Context Pruning

Bu notebook, tüm deney pipeline'ını uçtan uca çalıştırır.
- `SMOKE_TEST = True`  → T4 üzerinde hızlı duman testi (~birkaç dakika)
- `SMOKE_TEST = False` → A100 üzerinde tam deney

In [ ]:
# ============================================================
# SMOKE TEST FLAG  –  T4 için True, A100 tam koşum için False
# ============================================================
SMOKE_TEST = True
SMOKE_FLAG = "--smoke_test" if SMOKE_TEST else ""

## 1. Ortam Kurulumu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_DIR = '/content/CENG467_Final'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Mrtuzy/CENG467_Final.git $REPO_DIR
else:
    print(f'{REPO_DIR} zaten mevcut, git pull yapılıyor...')
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!pwd

In [ ]:
!pip install -q -r requirements.txt
!pip install -q matplotlib

In [ ]:
# HuggingFace Token (LLaMA-3 erişimi için)
# Yöntem A: Colab Secrets'a 'HF_TOKEN' ekleyin
# Yöntem B: Aşağıdaki satırı düzenleyin:
# import os; os.environ['HF_TOKEN'] = 'hf_xxx'

## 2. Veri Hazırlığı (QReCC)

In [ ]:
!python src/data_prep.py $SMOKE_FLAG

## 3. Öğretmen Etiketleme (LLaMA-3-8B, Leave-One-Out)

In [ ]:
!python src/teacher_labeling.py $SMOKE_FLAG

# GPU belleği temizle
import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 4. SBERT Vektörizasyon + DistilGPT-2 Entropi → Δt

In [ ]:
!python src/build_inputs.py $SMOKE_FLAG

import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 5. CfC Ağı Eğitimi

In [ ]:
!python src/train_cfc.py $SMOKE_FLAG

In [ ]:
# Training loss grafiğini göster
from IPython.display import Image, display
import os
fig_path = '/content/drive/MyDrive/CENG_467/figures/training_loss.png'
if os.path.exists(fig_path):
    display(Image(filename=fig_path, width=600))
else:
    print('Training loss grafiği bulunamadı.')

## 6. Değerlendirme (Full / CfC / Random / Cosine)

In [ ]:
!python src/evaluate.py $SMOKE_FLAG

import torch; torch.cuda.empty_cache()
import gc; gc.collect()

## 7. Sonuçlar ve Grafikler

In [ ]:
import json, os
results_path = '/content/drive/MyDrive/CENG_467/outputs/eval_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    # Tabloyu göster
    print(f"{'Method':<12} {'ROUGE-L':>10} {'TTFT (s)':>10} {'Tokens':>10} {'Reduction%':>12}")
    print('-'*58)
    for m, v in results.items():
        print(f"{m:<12} {v['ROUGE-L']:>10.4f} {v['TTFT']:>10.4f} {v['Avg_Tokens']:>10.1f} {v['Reduction%']:>11.1f}%")
else:
    print('Sonuç dosyası bulunamadı.')

In [ ]:
# Tüm grafikleri göster
from IPython.display import Image, display
import glob

fig_dir = '/content/drive/MyDrive/CENG_467/figures'
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))

if figs:
    for fp in figs:
        print(f'\n📊 {os.path.basename(fp)}')
        display(Image(filename=fp, width=600))
else:
    print('Grafik bulunamadı.')

---
### ✅ Deney tamamlandı!

Tüm sonuçlar ve grafikler Google Drive'da:
- `MyDrive/CENG_467/outputs/eval_results.json`
- `MyDrive/CENG_467/figures/*.png`
- `MyDrive/CENG_467/models/best_cfc_model.pth`